In [1]:
import pandas as pd

file_path = "data/yellow_tripdata_2021-01.csv.gz"

df = pd.read_csv(file_path, nrows=100)

In [2]:
print(df.head())

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


In [3]:
print(df.dtypes)

VendorID                   int64
tpep_pickup_datetime         str
tpep_dropoff_datetime        str
passenger_count            int64
trip_distance            float64
RatecodeID                 int64
store_and_fwd_flag           str
PULocationID               int64
DOLocationID               int64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
congestion_surcharge     float64
dtype: object


### Step 1 – Initial Data Inspection

Each row represents one NYC yellow taxi trip. The dataset contains trip timestamps, passenger information, pickup and drop-off locations, trip distance, fare details, and payment information.

The date columns are `tpep_pickup_datetime` and `tpep_dropoff_datetime`. They were initially read as strings (`str`) rather than datetime values.

No missing values are obvious from the first five displayed rows, so the full dataset must be profiled to identify missing values accurately.

In [8]:
total_rows = 0

null_counts = None
min_values = {}
max_values = {}
column_dtypes = None

for chunk in pd.read_csv(file_path, chunksize=100_000, low_memory=False):

    total_rows += len(chunk)

    if null_counts is None:
        null_counts = chunk.isna().sum()
        column_dtypes = chunk.dtypes.astype(str)

    else:
        null_counts += chunk.isna().sum()

    for col in chunk.columns:

        values = chunk[col].dropna()

        if len(values) == 0:
            continue

        if pd.api.types.is_string_dtype(chunk[col]):
            values = values.astype("string")

        chunk_min = values.min()
        chunk_max = values.max()

        if col not in min_values or chunk_min < min_values[col]:
            min_values[col] = chunk_min

        if col not in max_values or chunk_max > max_values[col]:
            max_values[col] = chunk_max

In [9]:
profile_df = pd.DataFrame({
    "dtype": column_dtypes,
    "null_count": null_counts,
    "null_percent": (null_counts / total_rows * 100).round(2),
    "min": pd.Series(min_values),
    "max": pd.Series(max_values)
})

print(profile_df)

                         dtype  null_count  null_percent                  min  \
VendorID                 int64       98352          7.18                    1   
tpep_pickup_datetime       str           0          0.00  2008-12-31 23:05:14   
tpep_dropoff_datetime      str           0          0.00  2008-12-31 23:07:22   
passenger_count          int64       98352          7.18                    0   
trip_distance          float64           0          0.00                  0.0   
RatecodeID               int64       98352          7.18                    1   
store_and_fwd_flag         str       98352          7.18                    N   
PULocationID             int64           0          0.00                    1   
DOLocationID             int64           0          0.00                    1   
payment_type             int64       98352          7.18                    1   
fare_amount            float64           0          0.00               -490.0   
extra                  float

In [10]:
print(profile_df.to_string())

                         dtype  null_count  null_percent                  min                  max
VendorID                 int64       98352          7.18                    1                    2
tpep_pickup_datetime       str           0          0.00  2008-12-31 23:05:14  2021-02-22 16:52:16
tpep_dropoff_datetime      str           0          0.00  2008-12-31 23:07:22  2021-02-22 16:56:15
passenger_count          int64       98352          7.18                    0                    8
trip_distance          float64           0          0.00                  0.0            263163.28
RatecodeID               int64       98352          7.18                    1                   99
store_and_fwd_flag         str       98352          7.18                    N                    Y
PULocationID             int64           0          0.00                    1                  265
DOLocationID             int64           0          0.00                    1                  265
payment_ty

In [11]:
import numpy as np

total_rows = 0

outside_january = 0
negative_fares = 0
zero_fares = 0
negative_distances = 0
zero_distances = 0
zero_passengers = 0

duplicate_rows = 0
seen_hashes = set()

for chunk in pd.read_csv(file_path, chunksize=100_000, low_memory=False):

    total_rows += len(chunk)

    # Convert pickup dates temporarily to datetime
    pickup_dates = pd.to_datetime(
        chunk["tpep_pickup_datetime"],
        errors="coerce"
    )

    # Trips picked up outside January 2021
    outside_january += (
        (pickup_dates < "2021-01-01") |
        (pickup_dates >= "2021-02-01") |
        (pickup_dates.isna())
    ).sum()

    # Invalid fares
    negative_fares += (chunk["fare_amount"] < 0).sum()
    zero_fares += (chunk["fare_amount"] == 0).sum()

    # Invalid distances
    negative_distances += (chunk["trip_distance"] < 0).sum()
    zero_distances += (chunk["trip_distance"] == 0).sum()

    # Zero passengers
    zero_passengers += (chunk["passenger_count"] == 0).sum()

    # Detect duplicate rows across all chunks
    hashes = pd.util.hash_pandas_object(
        chunk,
        index=False
    ).to_numpy()

    unique_hashes, counts = np.unique(
        hashes,
        return_counts=True
    )

    duplicate_rows += int((counts - 1).sum())

    duplicate_rows += sum(
        int(h) in seen_hashes for h in unique_hashes
    )

    seen_hashes.update(unique_hashes.tolist())


print("Total rows:", total_rows)
print("Pickups outside January:", outside_january)
print("Negative fares:", negative_fares)
print("Zero fares:", zero_fares)
print("Negative distances:", negative_distances)
print("Zero distances:", zero_distances)
print("Zero passengers:", zero_passengers)
print("Duplicate rows:", duplicate_rows)

Total rows: 1369765
Pickups outside January: 24
Negative fares: 6769
Zero fares: 642
Negative distances: 0
Zero distances: 19952
Zero passengers: 26726
Duplicate rows: 0


### Step 3 – Data Quality Findings

The dataset contains 1,369,765 rows.

The inspection identified:
- 24 pickup records outside January 2021.
- 6,769 records with negative fares.
- 642 records with zero fares.
- 19,952 records with zero trip distance.
- 26,726 records with zero passengers.
- 98,352 missing values in each of five columns.
- No negative trip distances or duplicate rows were detected.

These findings will be evaluated before applying any cleaning rules.

### Step 4 – Cleaning Decisions

| Problem | Decision | Reason |
|---|---|---|
| Pickups outside January 2021 | Drop | These records are outside the required reporting period. |
| Negative fares | Flag | Negative fares may represent refunds or adjustments. |
| Zero fares | Flag | Zero fares may represent cancelled or special trips. |
| Zero trip distance | Flag | These records require further investigation. |
| Zero passengers | Flag | These records may indicate missing or invalid passenger information. |
| Missing values | Flag | Records are preserved until business rules determine how to handle missing values. |
| Duplicate rows | No action | No duplicate rows were detected. |
| Negative distances | No action | No negative distances were detected. |

The Data Engineer implements the cleaning rules, while the Data Analyst and business team validate unusual values and approve business-specific decisions.

In [12]:
full_df = pd.read_csv(file_path, low_memory=False)

print("Original rows:", len(full_df))
print("Original shape:", full_df.shape)

Original rows: 1369765
Original shape: (1369765, 18)


In [15]:
def clean(df):

    cleaned_df = df.copy()
    rows_removed = {}

    # Check pickup dates
    pickup_dates = pd.to_datetime(
        cleaned_df["tpep_pickup_datetime"],
        errors="coerce"
    )

    invalid_dates = (
        (pickup_dates < "2021-01-01") |
        (pickup_dates >= "2021-02-01") |
        (pickup_dates.isna())
    )

    # Remove records outside January
    rows_removed["outside_january"] = int(invalid_dates.sum())

    cleaned_df = cleaned_df.loc[~invalid_dates].copy()

    # Add flags instead of deleting unusual records
    cleaned_df["negative_fare_flag"] = cleaned_df["fare_amount"] < 0
    cleaned_df["zero_fare_flag"] = cleaned_df["fare_amount"] == 0
    cleaned_df["zero_distance_flag"] = cleaned_df["trip_distance"] == 0
    cleaned_df["zero_passengers_flag"] = cleaned_df["passenger_count"] == 0

    # Flag rows containing missing values
    cleaned_df["missing_values_flag"] = cleaned_df.isna().any(axis=1)

    return cleaned_df, rows_removed

In [14]:
cleaned_df, rows_removed = clean(full_df)

print("Rows before cleaning:", len(full_df))
print("Rows after cleaning:", len(cleaned_df))
print("Rows removed:", rows_removed)

Rows before cleaning: 1369765
Rows after cleaning: 1369741
Rows removed: {'outside_january': 24}


In [16]:
cleaned_df, rows_removed = clean(full_df)

print("Rows before cleaning:", len(full_df))
print("Rows after cleaning:", len(cleaned_df))
print("Rows removed:", rows_removed)

print("\nFlag columns:")
print(cleaned_df[[
    "negative_fare_flag",
    "zero_fare_flag",
    "zero_distance_flag",
    "zero_passengers_flag",
    "missing_values_flag"
]].sum())

Rows before cleaning: 1369765
Rows after cleaning: 1369741
Rows removed: {'outside_january': 24}

Flag columns:
negative_fare_flag       6769
zero_fare_flag            641
zero_distance_flag      19951
zero_passengers_flag    26726
missing_values_flag     98352
dtype: int64


In [17]:
results = pd.DataFrame([
    ["Outside January", "Drop", 24, "Outside the required period"],
    ["Negative fares", "Flag", 6769, "May represent refunds or adjustments"],
    ["Zero fares", "Flag", 641, "May represent cancelled or special trips"],
    ["Zero distances", "Flag", 19951, "Require further investigation"],
    ["Zero passengers", "Flag", 26726, "Require further investigation"],
    ["Missing values", "Flag", 98352, "Preserve records for review"],
    ["Negative distances", "No action", 0, "No records found"],
    ["Duplicate rows", "No action", 0, "No records found"]
], columns=["Problem", "Decision", "Rows affected", "Reason"])

print(results.to_string(index=False))

           Problem  Decision  Rows affected                                   Reason
   Outside January      Drop             24              Outside the required period
    Negative fares      Flag           6769     May represent refunds or adjustments
        Zero fares      Flag            641 May represent cancelled or special trips
    Zero distances      Flag          19951            Require further investigation
   Zero passengers      Flag          26726            Require further investigation
    Missing values      Flag          98352              Preserve records for review
Negative distances No action              0                         No records found
    Duplicate rows No action              0                         No records found


In [18]:
print("Final Cleaning Validation")

print("Original rows:", len(full_df))
print("Cleaned rows:", len(cleaned_df))
print("Total rows removed:", len(full_df) - len(cleaned_df))

print("\nRows removed by each rule:")
print(rows_removed)

print("\nOriginal data unchanged:")
print(len(full_df) == 1369765)

Final Cleaning Validation
Original rows: 1369765
Cleaned rows: 1369741
Total rows removed: 24

Rows removed by each rule:
{'outside_january': 24}

Original data unchanged:
True
